# SpikeLoRA on isiZulu Sentiment (mDeBERTaV3-Base)

Fine-tunes [mDeBERTaV3-Base](https://huggingface.co/microsoft/mdeberta-v3-base) for sentiment classification on `data/zulu_tweets.csv`, with plain LoRA and with SpikeLoRA.

### Dataset

We use the isiZulu slice of SAfriSenti-Corpus-Mini:

```bibtex
@misc{mabokela2026safrisenti_mini,
  author       = {Mabokela, Koena Ronny and Primus, Mpho and Celik, Turgay},
  title        = {{SAfriSenti-Corpus-Mini}: A Mini Multilingual Sentiment Corpus for South African Under-Resourced Languages},
  year         = {2026},
  howpublished = {GitHub repository},
  note         = {Balanced sample containing 1,500 labelled sentiment records across Sepedi, Sesotho, Setswana, isiXhosa, and isiZulu},
  url          = {https://github.com/NLPforLRLsProjects/SAfriSenti-Corpus}
}
```

Labels: positive, neutral, negative

## Setup

In [1]:
import sys

!{sys.executable} -m pip install -q -e ../..
!{sys.executable} -m pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [ ]:
import json
import math
import os
from typing import Iterable, List, Optional

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, EvalPrediction, Trainer, TrainingArguments, set_seed

import spikeloralib as lora
from spikeloralib import spikelora

## LoRA / SpikeLoRA injection for mDeBERTa

In [3]:
DEBERTA_TARGET_MODULES = [
    "query_proj",
    "key_proj",
    "value_proj",
    "attention.output.dense",
    "intermediate.dense",
    "output.dense",
]


def _get_parent(model: nn.Module, dotted_name: str) -> nn.Module:
    if "." not in dotted_name:
        return model
    return model.get_submodule(dotted_name.rsplit(".", 1)[0])


def _replace_linear(model: nn.Module, name: str, old: nn.Linear, layer_cls, **kwargs) -> None:
    new = layer_cls(old.in_features, old.out_features, bias=old.bias is not None, **kwargs)
    new = new.to(dtype=old.weight.dtype, device=old.weight.device)
    with torch.no_grad():
        new.weight.copy_(old.weight)
        if old.bias is not None:
            new.bias.copy_(old.bias)
    parent = _get_parent(model, name)
    setattr(parent, name.rsplit(".", 1)[-1], new)


def apply_lora(
    model: nn.Module,
    r: int,
    lora_alpha: int,
    lora_dropout: float = 0.0,
    use_rslora: bool = False,
    use_spikelora: bool = False,
    v_threshold: float = 1.0,
    tau: float = 2.0,
    target_modules: Optional[Iterable[str]] = None,
) -> List[str]:
    target_modules = list(target_modules) if target_modules is not None else DEBERTA_TARGET_MODULES
    layer_cls = spikelora.Linear if use_spikelora else lora.Linear
    extra_kwargs = {"v_threshold": v_threshold, "tau": tau} if use_spikelora else {}

    replaced = []
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and any(t in name for t in target_modules):
            _replace_linear(
                model, name, module, layer_cls,
                r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout, use_rslora=use_rslora,
                **extra_kwargs,
            )
            replaced.append(name)
    return replaced


def mark_trainable(model: nn.Module, train_classifier_substrings: Iterable[str] = ("classifier", "pooler")) -> None:
    lora.mark_only_lora_as_trainable(model)
    for name, param in model.named_parameters():
        if any(s in name for s in train_classifier_substrings):
            param.requires_grad = True


def spikelora_modules(model: nn.Module):
    return [m for m in model.modules() if isinstance(m, spikelora.Linear)]


def average_sparsity(model: nn.Module) -> Optional[float]:
    values = [m.sparsity for m in spikelora_modules(model) if m.sparsity is not None]
    return sum(values) / len(values) if values else None


def module_sparsity(model: nn.Module) -> dict:
    return {
        name: module.sparsity
        for name, module in model.named_modules()
        if isinstance(module, spikelora.Linear) and module.sparsity is not None
    }

## Trainer setup

In [4]:
class SpikeLoRATrainer(Trainer):
    def evaluate(self, *args, **kwargs):
        metrics = super().evaluate(*args, **kwargs)
        prefix = kwargs.get("metric_key_prefix", "eval")

        sparsity = average_sparsity(self.model)
        if sparsity is not None:
            metrics[f"{prefix}_sparsity"] = sparsity

        train_loss = self._last_train_loss()
        loss_key = f"{prefix}_loss"
        if train_loss is not None and loss_key in metrics:
            metrics[f"{prefix}_generalisation_gap"] = metrics[loss_key] - train_loss

        self.log(metrics)
        return metrics

    def _last_train_loss(self):
        for entry in reversed(self.state.log_history):
            if "loss" in entry:
                return entry["loss"]
        return None

## Dataset loading

In [5]:
DATA_PATH = os.path.join("data", "zulu_tweets.csv")
TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"
MODEL_NAME = "microsoft/mdeberta-v3-base"
MAX_SEQ_LENGTH = 128
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 0


def load_splits(data_path, label_column, test_size, val_size, seed):
    raw = load_dataset("csv", data_files=data_path)["train"]
    raw = raw.class_encode_column(label_column)
    label_names = raw.features[label_column].names

    train_val_test = raw.train_test_split(test_size=test_size, stratify_by_column=label_column, seed=seed)
    val_fraction_of_remainder = val_size / (1.0 - test_size)
    train_val = train_val_test["train"].train_test_split(
        test_size=val_fraction_of_remainder, stratify_by_column=label_column, seed=seed,
    )
    return train_val["train"], train_val["test"], train_val_test["test"], label_names


def preprocess(text_column, tokenizer, max_seq_length):
    def fn(examples):
        return tokenizer(examples[text_column], truncation=True, max_length=max_seq_length)

    return fn


def build_compute_metrics():
    def compute_metrics(p: EvalPrediction):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        preds = np.argmax(preds, axis=1)
        return {
            "accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro"),
            "f1_weighted": f1_score(p.label_ids, preds, average="weighted"),
        }

    return compute_metrics

In [6]:
set_seed(SEED)

train_raw, val_raw, test_raw, label_names = load_splits(DATA_PATH, LABEL_COLUMN, TEST_SIZE, VAL_SIZE, SEED)
num_labels = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
preprocess_fn = preprocess(TEXT_COLUMN, tokenizer, MAX_SEQ_LENGTH)

train_dataset = train_raw.map(preprocess_fn, batched=True, remove_columns=[TEXT_COLUMN]).rename_column(LABEL_COLUMN, "label")
eval_dataset = val_raw.map(preprocess_fn, batched=True, remove_columns=[TEXT_COLUMN]).rename_column(LABEL_COLUMN, "label")
test_dataset = test_raw.map(preprocess_fn, batched=True, remove_columns=[TEXT_COLUMN]).rename_column(LABEL_COLUMN, "label")

Map:   0%|          | 0/2099 [00:00<?, ? examples/s]

Map:   0%|          | 0/451 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

## Train/eval helper (one LoRA or SpikeLoRA run)

In [7]:
def train_and_eval(
    output_dir,
    use_spikelora=False,
    v_threshold=0.1,
    tau=2.0,
    lora_r=8,
    lora_alpha=None,
    lora_dropout=0.0,
    use_rslora=True,
    learning_rate=3e-4,
    num_train_epochs=15,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.06,
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=0,
    max_steps=-1,
):
    if lora_alpha is None:
        lora_alpha = math.sqrt(lora_r) if use_rslora else float(lora_r)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels,
        id2label={i: name for i, name in enumerate(label_names)},
        label2id={name: i for i, name in enumerate(label_names)},
        torch_dtype=torch.float32,
    )

    apply_lora(
        model, r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
        use_rslora=use_rslora, use_spikelora=use_spikelora, v_threshold=v_threshold, tau=tau,
    )
    mark_trainable(model)

    steps_per_epoch = math.ceil(len(train_dataset) / per_device_train_batch_size)
    total_steps = max_steps if max_steps > 0 else int(steps_per_epoch * num_train_epochs)
    warmup_steps = max(1, int(total_steps * warmup_ratio))

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        max_steps=max_steps,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        max_grad_norm=max_grad_norm,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=25,
        seed=seed,
        report_to=[],
    )

    trainer = SpikeLoRATrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=build_compute_metrics(),
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")

    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "eval_metrics.json"), "w") as f:
        json.dump(eval_metrics, f, indent=2)
    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump(test_metrics, f, indent=2)
    if use_spikelora:
        with open(os.path.join(output_dir, "module_sparsity.json"), "w") as f:
            json.dump(module_sparsity(model), f, indent=2)

    return eval_metrics, test_metrics

## Run: plain LoRA

In [8]:
lora_eval_metrics, lora_test_metrics = train_and_eval(
    output_dir="runs/zulu-lora",
    use_spikelora=False,
    lora_r=8,
    use_rslora=True,
    learning_rate=3e-4,
    num_train_epochs=15,
)
lora_eval_metrics, lora_test_metrics

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.33GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

...

/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.400348,0.837879,15,0.687361,0.601347,0.679637


/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.400348,0.831017,15,0.691111,0.583158,0.683513


({'eval_loss': 0.8378785848617554,
  'eval_accuracy': 0.6873614190687362,
  'eval_f1_macro': 0.6013471758744283,
  'eval_f1_weighted': 0.6796370608058391,
  'eval_generalisation_gap': 0.43753045558929443,
  'epoch': 15.0},
 {'test_loss': 0.8310169577598572,
  'test_accuracy': 0.6911111111111111,
  'test_f1_macro': 0.5831583506382517,
  'test_f1_weighted': 0.6835130595740349,
  'test_generalisation_gap': 0.43066882848739624,
  'epoch': 15.0})

## Run: SpikeLoRA

In [9]:
spikelora_eval_metrics, spikelora_test_metrics = train_and_eval(
    output_dir="runs/zulu-spikelora",
    use_spikelora=True,
    v_threshold=0.1,
    lora_r=8,
    use_rslora=True,
    learning_rate=3e-4,
    num_train_epochs=15,
)
spikelora_eval_metrics, spikelora_test_metrics

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.937247,0.892529,0.549889,0.350989,0.499643
2,0.842246,0.886289,0.538803,0.368781,0.516893
3,0.758593,0.809852,0.614191,0.425778,0.592351
4,0.730197,0.798051,0.631929,0.486898,0.612534
5,0.674547,0.754849,0.669623,0.459927,0.643108
6,0.621786,0.738694,0.678492,0.466482,0.652056
7,0.568551,0.746230,0.685144,0.503618,0.665063
8,0.528523,0.761138,0.678492,0.570678,0.666144
9,0.532540,0.927103,0.669623,0.533938,0.645177
10,0.457101,0.759138,0.694013,0.623139,0.688403


/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.396874,0.854223,15,0.713969,0.643291,0.707024


/Users/iwandejong/.pyenv/versions/3.10.11/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.396874,0.847255,15,0.702222,0.597314,0.694385


({'eval_loss': 0.8542228937149048,
  'eval_accuracy': 0.7139689578713969,
  'eval_f1_macro': 0.6432910282348333,
  'eval_f1_weighted': 0.7070237237692507,
  'eval_sparsity': 0.6717507217585484,
  'eval_generalisation_gap': 0.4573493814468384,
  'epoch': 15.0},
 {'test_loss': 0.847255289554596,
  'test_accuracy': 0.7022222222222222,
  'test_f1_macro': 0.5973142562154422,
  'test_f1_weighted': 0.6943850870155923,
  'test_sparsity': 0.650720731675392,
  'test_generalisation_gap': 0.45038177728652956,
  'epoch': 15.0})

## Results

In [10]:
{
    "lora": {"eval": lora_eval_metrics, "test": lora_test_metrics},
    "spikelora": {"eval": spikelora_eval_metrics, "test": spikelora_test_metrics},
}

{'lora': {'eval': {'eval_loss': 0.8378785848617554,
   'eval_accuracy': 0.6873614190687362,
   'eval_f1_macro': 0.6013471758744283,
   'eval_f1_weighted': 0.6796370608058391,
   'eval_generalisation_gap': 0.43753045558929443,
   'epoch': 15.0},
  'test': {'test_loss': 0.8310169577598572,
   'test_accuracy': 0.6911111111111111,
   'test_f1_macro': 0.5831583506382517,
   'test_f1_weighted': 0.6835130595740349,
   'test_generalisation_gap': 0.43066882848739624,
   'epoch': 15.0}},
 'spikelora': {'eval': {'eval_loss': 0.8542228937149048,
   'eval_accuracy': 0.7139689578713969,
   'eval_f1_macro': 0.6432910282348333,
   'eval_f1_weighted': 0.7070237237692507,
   'eval_sparsity': 0.6717507217585484,
   'eval_generalisation_gap': 0.4573493814468384,
   'epoch': 15.0},
  'test': {'test_loss': 0.847255289554596,
   'test_accuracy': 0.7022222222222222,
   'test_f1_macro': 0.5973142562154422,
   'test_f1_weighted': 0.6943850870155923,
   'test_sparsity': 0.650720731675392,
   'test_generalisation